# DSN hybrid: train-only retrieval + classifier + BBC headline model

This notebook never reads `masakhane-news/data/*/test.tsv` or any labeled test source. It does not submit to the competition. It writes `submission_hybrid.csv`, checkpoints, and a validation report. Training may take longer than 30 minutes; subsequent inference from checkpoints is intended to fit that window.


In [ ]:
%pip uninstall -y torchao
%pip install -q 'transformers>=4.45,<5' 'peft>=0.13,<1' datasets accelerate sentencepiece rouge-score


In [ ]:
"""Train-only hybrid solution for DSN 2026. Never loads labeled test TSVs.

The CSV and model checkpoints are outputs; no Kaggle submission API is called.
"""

from __future__ import annotations

import gc
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC


LABELS = ["business", "health", "politics", "religion", "sports", "entertainment", "technology"]
MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"


def clean(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def key(value):
    return clean(value).casefold()


def split_indices(train, seed=42):
    strata = train.lang.astype(str) + "_" + train.category.astype(str)
    groups = train.text.map(key)
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    fit_idx, valid_idx = next(splitter.split(train, strata, groups))
    assert not set(groups.iloc[fit_idx]) & set(groups.iloc[valid_idx])
    return fit_idx, valid_idx


def topic_features(fit_text, other_text):
    word = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2,
                           max_features=180_000, strip_accents="unicode")
    char = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), sublinear_tf=True,
                           min_df=3, max_features=300_000, strip_accents="unicode")
    fit_word = word.fit_transform(fit_text)
    fit_char = char.fit_transform(fit_text)
    other_word = word.transform(other_text)
    other_char = char.transform(other_text)
    return hstack([fit_word, fit_char]).tocsr(), hstack([other_word, other_char]).tocsr()


def fit_topic(fit, other):
    x_fit, x_other = topic_features(fit.text.fillna("").map(clean),
                                     other.text.fillna("").map(clean))
    model = LinearSVC(C=0.5, class_weight="balanced", random_state=42)
    model.fit(x_fit, fit.category)
    return model.classes_, model.decision_function(x_other)


def predict_topic(classes, scores, langs, allowed=None):
    scores = np.asarray(scores).copy()
    if allowed is not None:
        for row, language in enumerate(langs):
            for col, label in enumerate(classes):
                if label not in allowed[language]:
                    scores[row, col] = -1e9
    return np.asarray(classes)[scores.argmax(axis=1)]


def tune_topic(fit, valid):
    classes, scores = fit_topic(fit, valid)
    allowed = fit.groupby("lang").category.agg(lambda values: set(values)).to_dict()
    raw = predict_topic(classes, scores, valid.lang)
    masked = predict_topic(classes, scores, valid.lang, allowed)
    raw_f1 = f1_score(valid.category, raw, labels=LABELS, average="macro")
    masked_f1 = f1_score(valid.category, masked, labels=LABELS, average="macro")
    use_mask = masked_f1 > raw_f1
    result = masked if use_mask else raw
    report = {"topic_f1_unmasked": float(raw_f1),
              "topic_f1_masked": float(masked_f1),
              "topic_mask_selected": use_mask,
              "topic_f1_selected": float(max(raw_f1, masked_f1))}
    return result, report


def retrieval(fit, query):
    """Exact and high-similarity search over training text only."""
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5),
                                 min_df=2, max_features=300_000, sublinear_tf=True)
    fit_text = fit.text.fillna("").map(lambda value: clean(value)[:1600])
    query_text = query.text.fillna("").map(lambda value: clean(value)[:1600])
    x_fit = vectorizer.fit_transform(fit_text)
    x_query = vectorizer.transform(query_text)
    exact = {}
    for pos, value in enumerate(fit.text):
        exact.setdefault(key(value), pos)
    indices, similarities, exact_count = [], [], 0
    for start in range(0, len(query), 64):
        matrix = (x_query[start:start + 64] @ x_fit.T).tocsr()
        for offset in range(matrix.shape[0]):
            row = matrix.getrow(offset)
            local_pos = start + offset
            match = exact.get(key(query.text.iloc[local_pos]))
            if match is not None:
                indices.append(match)
                similarities.append(1.0)
                exact_count += 1
            elif row.nnz:
                best = row.data.argmax()
                indices.append(int(row.indices[best]))
                similarities.append(float(row.data[best]))
            else:
                indices.append(0)
                similarities.append(0.0)
    return np.asarray(indices), np.asarray(similarities), exact_count


def headline_prompt(row, topic):
    # The pretrained model is already a BBC-news summarizer; short control tokens
    # are less disruptive than a verbose English instruction.
    return f"headline {row.lang} topic {topic}: {clean(row.text)}"


def train_headline(fit, directory: Path, epochs=2):
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,
                              DataCollatorForSeq2Seq, Seq2SeqTrainer,
                              Seq2SeqTrainingArguments)

    directory.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    assert sum(param.numel() for param in base.parameters()) < 1_000_000_000
    model = get_peft_model(base, LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM,
                                             r=16, lora_alpha=32, lora_dropout=0.05,
                                             target_modules=["q", "v"]))
    dataset = Dataset.from_dict({
        "source": [headline_prompt(row, row.category) for row in fit.itertuples(index=False)],
        "target": fit.headline.map(clean).tolist(),
    })

    def tokenize(batch):
        source = tokenizer(batch["source"], max_length=384, truncation=True)
        target = tokenizer(text_target=batch["target"], max_length=56, truncation=True)
        source["labels"] = target["input_ids"]
        return source

    dataset = dataset.map(tokenize, batched=True, remove_columns=["source", "target"])
    args = Seq2SeqTrainingArguments(
        output_dir=str(directory / "tmp"), num_train_epochs=epochs,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.06,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        fp16=True, logging_steps=50, save_strategy="no", eval_strategy="no",
        report_to="none", predict_with_generate=False, seed=42,
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=dataset,
                             processing_class=tokenizer,
                             data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
    trainer.train()
    model.save_pretrained(directory)
    tokenizer.save_pretrained(directory)
    del trainer, model, base, dataset
    gc.collect()
    torch.cuda.empty_cache()


def generate_headline(frame, topics, directory: Path, batch_size=8):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(directory)
    base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    model = PeftModel.from_pretrained(base, directory).cuda().eval()
    headlines = []
    for start in range(0, len(frame), batch_size):
        rows = frame.iloc[start:start + batch_size]
        prompts = [headline_prompt(row, topic) for row, topic in
                   zip(rows.itertuples(index=False), topics[start:start + batch_size])]
        inputs = tokenizer(prompts, max_length=384, truncation=True,
                           padding=True, return_tensors="pt").to("cuda")
        with torch.inference_mode():
            ids = model.generate(**inputs, max_new_tokens=56, min_new_tokens=5,
                                 num_beams=4, no_repeat_ngram_size=3,
                                 length_penalty=0.85)
        headlines.extend(clean(value) for value in tokenizer.batch_decode(ids, skip_special_tokens=True))
        if start % 400 == 0:
            print("Generated", start, "of", len(frame), flush=True)
    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    return headlines


def rouge_l(reference, prediction):
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    return scorer.score(clean(reference), clean(prediction))["rougeL"].fmeasure


def tune_retrieval(fit, valid, model_headlines, topic_predictions):
    index, similarity, exact_count = retrieval(fit, valid)
    baseline = np.array([rouge_l(a, b) for a, b in zip(valid.headline, model_headlines)])
    report = {"headline_rouge_model": float(baseline.mean()),
              "validation_exact_matches": int(exact_count)}
    best = (float(baseline.mean()), 1.01)
    for threshold in [0.80, 0.85, 0.90, 0.93, 0.96, 0.99, 1.00]:
        candidates = []
        for row, match, score, generated in zip(valid.itertuples(index=False), index,
                                                 similarity, model_headlines):
            candidates.append(fit.headline.iloc[match] if score >= threshold else generated)
        score = float(np.mean([rouge_l(a, b) for a, b in zip(valid.headline, candidates)]))
        report[f"headline_rouge_retrieval_{threshold:.2f}"] = score
        if score > best[0]:
            best = score, threshold
    report["retrieval_threshold"] = float(best[1])
    report["headline_rouge_selected"] = float(best[0])
    return report


def make_submission(train, test, topics, headlines, threshold, destination):
    index, similarity, exact_count = retrieval(train, test)
    exact_keys = set(train.text.map(key))
    topics = list(topics)
    headlines = list(headlines)
    used = 0
    for pos, (match, score) in enumerate(zip(index, similarity)):
        if key(test.text.iloc[pos]) in exact_keys or score >= threshold:
            topics[pos] = clean(train.category.iloc[match])
            headlines[pos] = clean(train.headline.iloc[match])
            used += 1
    rows = []
    for row, topic, headline in zip(test.itertuples(index=False), topics, headlines):
        assert topic in LABELS and headline
        rows.extend([(f"{row.id}_topic", topic), (f"{row.id}_headline", headline)])
    submission = pd.DataFrame(rows, columns=["id", "prediction"])
    submission.to_csv(destination, index=False)
    return {"test_exact_matches": exact_count, "test_retrieval_used": used,
            "submission_rows": len(submission)}


In [ ]:
from pathlib import Path
import json, time
from sklearn.metrics import f1_score
import pandas as pd
root = next(p.parent for p in Path('/kaggle/input').glob('**/train.csv') if (p.parent/'test.csv').exists() and (p.parent/'sample_submission.csv').exists())
train = pd.read_csv(root/'train.csv').fillna('')
test = pd.read_csv(root/'test.csv').fillna('')
sample = pd.read_csv(root/'sample_submission.csv')
fit_idx, valid_idx = split_indices(train)
fit = train.iloc[fit_idx].reset_index(drop=True)
valid = train.iloc[valid_idx].reset_index(drop=True)
print('train', len(train), 'fit', len(fit), 'valid', len(valid), 'test', len(test))


In [ ]:
valid_topics, topic_report = tune_topic(fit, valid)
print('Topic validation:', topic_report)
classes, test_scores = fit_topic(train, test)
allowed = train.groupby('lang').category.agg(lambda values: set(values)).to_dict()
test_topics = predict_topic(classes, test_scores, test.lang, allowed if topic_report['topic_mask_selected'] else None)
print(pd.Series(test_topics).value_counts())


In [ ]:
headline_dir = Path('/kaggle/working/hybrid_headline_adapter')
train_headline(fit, headline_dir, epochs=2)
valid_headlines = generate_headline(valid, valid_topics, headline_dir)
headline_report = tune_retrieval(fit, valid, valid_headlines, valid_topics)
print('Headline validation:', headline_report)
report = {**topic_report, **headline_report}
Path('hybrid_validation.json').write_text(json.dumps(report, indent=2))


In [ ]:
test_headlines = generate_headline(test, test_topics, headline_dir)
build_report = make_submission(train, test, test_topics, test_headlines, headline_report['retrieval_threshold'], Path('submission_hybrid.csv'))
result = pd.read_csv('submission_hybrid.csv')
assert result.id.tolist() == sample.id.tolist()
assert result.prediction.astype(str).str.len().gt(0).all()
print('Build report:', build_report)
print('Valid submission shape:', result.shape)
